# HumAID — Zero-shot Dev-Split Prompt Sensitivity (RULES_1 / RULES_2 / RULES_3)

Runs GPT-4o on the **DEV split** of all 10 HumAID events with three prompt rule variants:

- **RULES_1** — compact single-line class definitions
- **RULES_2** — medium-detail with "PRIMARY INTENT" framing
- **RULES_3** — detailed Definition / Include / Exclude guidance per class

## Why dev split?
The earlier prompt comparison (notebooks 05 / 07 / 09) used the **test split**, which is methodologically wrong for model/prompt selection (test-set contamination). This notebook re-runs the comparison on the dev split so the numbers are safe to report in the paper.

## Key settings
- `MODEL = gpt-4o`, 3 rule variants run sequentially
- `BATCH_TOKEN_LIMIT = 5_000_000` (Tier 3+ sized; Tier 5 can go much higher but dev events are small so 5M is plenty)
- `OPENAI_API_KEY_2` via `use_api_key_env` context manager (user's own Tier-3+ key — KEY_1 is professor's Tier-1 key, do not use)
- Results saved under `runs/<event>/dev/gpt-4o/<timestamp>-<tag>/` with analysis charts per run
- Final summary CSV saved to `runs/_indexes/` and a pivot table printed at the end

# 0) Setup


In [ ]:
from pathlib import Path
import math
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv; load_dotenv()

from humaidclf import run_experiment
from humaidclf import build_token_index               # token budgeting (sampling-based)
from humaidclf import run_experiment_sharded          # stratified sharded runner
from humaidclf.batch import use_api_key_env           # key switcher
from rules import RULES_1, RULES_2, RULES_3

# --- config ---
BASE = Path("Dataset/HumAID")
SPLITS = ["dev"]               # <-- dev split only
MODEL = "gpt-4o"
TAG_BASE = "modeS-gpt-4o"
DRYRUN_N = 20
POLL_SECS = 300
DO_ANALYSIS = True
OUT_ROOT = "runs"
API_KEY_ENV_VAR = "OPENAI_API_KEY_2"   

# Token caps & estimates
# Tier-1: 90,000 | Tier-3: 5,000,000 | Tier-5: much higher
# 5M is plenty for dev split (largest dev event ~960 tweets, ~300k tokens)
BATCH_TOKEN_LIMIT = 5_000_000
SAFETY_MARGIN = 0.90
MAX_OUTPUT_TOKENS = 40

# Rule variants to run (in order)
RULES_VARIANTS = [
    ("RULES1", RULES_1),
    ("RULES2", RULES_2),
    ("RULES3", RULES_3),
]

# 1) Discover dev TSVs for all 10 events


In [ ]:
def discover_tsvs(base: Path, splits: list[str]):
    items = []
    for event_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        event = event_dir.name
        for split in splits:
            tsv = event_dir / f"{event}_{split}.tsv"
            if tsv.exists():
                items.append({"event": event, "split": split, "tsv": str(tsv)})
    return pd.DataFrame(items)

df_sources = discover_tsvs(BASE, SPLITS)
print(f"Found {len(df_sources)} dev TSV files")
display(df_sources)


# 2) Helper functions (single + sharded runners)


In [ ]:
def run_list_single(dflist: pd.DataFrame, rules_text: str, model: str, tag: str):
    """Run events that already fit under the cap using the normal runner."""
    results = []
    for _, row in dflist.iterrows():
        event, split, tsv = row["event"], row["split"], row["tsv"]
        print(f"\n=== Running (single) {event}/{split} ({model} | {tag}) ===")
        try:
            plan, preds, summary = run_experiment(
                dataset_path=tsv,
                rules=rules_text,
                model=model,
                tag=tag,
                dryrun_n=DRYRUN_N,
                poll_secs=POLL_SECS,
                out_root=OUT_ROOT,
                do_analysis=DO_ANALYSIS,
            )
            acc = summary.get("accuracy") if summary else float("nan")
            f1  = summary.get("macro_f1") if summary else float("nan")
            n   = summary.get("num_total_with_truth") if summary else len(preds)
            results.append({
                "event": event, "split": split,
                "run_dir": str(plan["dir"]),
                "predictions_csv": str(plan["predictions_csv"]),
                "macro_f1": f1, "accuracy": acc, "num_total": n,
                "mode": "single",
            })
        except Exception as e:
            print(f"[ERROR] {event}/{split}: {e}")
            results.append({
                "event": event, "split": split, "run_dir": "ERROR",
                "predictions_csv": "", "macro_f1": float("nan"),
                "accuracy": float("nan"), "num_total": 0, "mode": "single",
            })
    return pd.DataFrame(results)


def run_list_sharded(dflist: pd.DataFrame, token_df: pd.DataFrame, rules_text: str, model: str, tag: str):
    """Run events that exceed the cap using stratified shards. k is computed from token estimates."""
    results = []
    est_map = {(r.event, r.split): r.est_total_tokens for r in token_df.itertuples(index=False)}
    eff_cap = BATCH_TOKEN_LIMIT * SAFETY_MARGIN

    for _, row in dflist.iterrows():
        event, split, tsv = row["event"], row["split"], row["tsv"]
        est_tokens = est_map.get((event, split), None)
        k = max(2, math.ceil((est_tokens or (eff_cap + 1)) / eff_cap))
        print(f"\n=== Running (sharded x{k}) {event}/{split} ({model} | {tag}) ===")
        try:
            plan, preds, summary = run_experiment_sharded(
                dataset_path=tsv,
                rules=rules_text,
                model=model,
                tag=f"{tag}-sharded{k}",
                k_shards=k,
                temperature=0.0,
                poll_secs=POLL_SECS,
                out_root=OUT_ROOT,
                do_analysis=DO_ANALYSIS,
                analysis_subdir="analysis",
            )
            acc = summary.get("accuracy") if summary else float("nan")
            f1  = summary.get("macro_f1") if summary else float("nan")
            n   = summary.get("num_total_with_truth") if summary else len(preds)
            results.append({
                "event": event, "split": split,
                "run_dir": str(plan["dir"]),
                "predictions_csv": str(plan["predictions_csv"]),
                "macro_f1": f1, "accuracy": acc, "num_total": n,
                "mode": f"sharded{k}",
            })
        except Exception as e:
            print(f"[ERROR] {event}/{split}: {e}")
            results.append({
                "event": event, "split": split, "run_dir": "ERROR",
                "predictions_csv": "", "macro_f1": float("nan"),
                "accuracy": float("nan"), "num_total": 0, "mode": f"sharded{k}",
            })
    return pd.DataFrame(results)


# 3) Run all 3 rule variants on all 10 dev events

Loops over `(RULES1, RULES2, RULES3)`. For each variant:
1. Builds a token index sized for the current rules text
2. Splits events into `df_fit` (single batch) and `df_too_big` (sharded)
3. Runs both groups under the same API key context
4. Appends results tagged with the rule variant name

On Tier 5 with the 5M cap, every dev event should fit in a single batch, so `df_too_big` will typically be empty — but the sharded path stays as a safety net.


In [ ]:
all_runs_frames = []

for rules_name, rules_text in RULES_VARIANTS:
    print(f"\n\n{'='*80}")
    print(f" STARTING {rules_name}")
    print(f"{'='*80}\n")

    # Build token index for this rules variant
    token_index = build_token_index(
        df_sources,
        model=MODEL,
        rules_text=rules_text,
        batch_token_limit=BATCH_TOKEN_LIMIT,
        safety_margin=SAFETY_MARGIN,
        sample_size=200,
        max_output_tokens=MAX_OUTPUT_TOKENS,
    )
    print(f"Token index for {rules_name}:")
    display(token_index[["event", "split", "num_rows", "est_total_tokens", "limit_used_%", "fits_cap"]])

    df_fit     = token_index[token_index["fits_cap"]].reset_index(drop=True)
    df_too_big = token_index[~token_index["fits_cap"]].reset_index(drop=True)

    rules_tag = f"{TAG_BASE}-{rules_name}-filtered"

    with use_api_key_env(API_KEY_ENV_VAR):
        print(f">>> Using {API_KEY_ENV_VAR} for {rules_name}")

        # Single-batch events
        if not df_fit.empty:
            df_runs_single = run_list_single(df_fit, rules_text, MODEL, tag=rules_tag)
        else:
            df_runs_single = pd.DataFrame()

        # Sharded fallback (should be empty with 5M cap on dev split)
        if not df_too_big.empty:
            df_runs_sharded = run_list_sharded(df_too_big, token_index, rules_text, MODEL, tag=rules_tag)
        else:
            df_runs_sharded = pd.DataFrame()
            print("No large datasets to shard.")

    # Tag with the rules variant and accumulate
    runs_df = pd.concat([df_runs_single, df_runs_sharded], ignore_index=True)
    runs_df["rules"] = rules_name
    all_runs_frames.append(runs_df)

all_runs = pd.concat(all_runs_frames, ignore_index=True)
print(f"\n\nAll runs complete. Total: {len(all_runs)} (expected: {len(df_sources) * len(RULES_VARIANTS)})")
display(all_runs)


# 4) Save summary CSV and display pivot table


In [ ]:
# Save the combined run index
idx_dir = Path(OUT_ROOT) / "_indexes"
idx_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
out_csv = idx_dir / f"runs_{MODEL}_dev_3rules_{stamp}.csv"
all_runs.to_csv(out_csv, index=False)
print(f"Saved run index: {out_csv}")

# Pivot: rows = event, columns = rule variant, values = macro_f1
pivot = all_runs.pivot_table(index="event", columns="rules", values="macro_f1", aggfunc="first")
# Reorder columns to match RULES_VARIANTS order
col_order = [name for name, _ in RULES_VARIANTS if name in pivot.columns]
pivot = pivot[col_order]
pivot["row_best"] = pivot.max(axis=1)
print("\nMacro-F1 per event per rule variant:")
display(pivot.round(3))

# Averaged per rule variant
print("\nAveraged Macro-F1 across 10 events (per rule variant):")
means = all_runs.groupby("rules")["macro_f1"].mean().reindex([name for name, _ in RULES_VARIANTS])
print(means.round(3).to_string())

# Which variant wins?
best = means.idxmax()
print(f"\nBest variant: {best} (mean macro-F1 = {means[best]:.3f})")


# 5) Re-run any ERROR entries (manual)

If any rows above show `run_dir == 'ERROR'`, fill `rerun_events` with their event names and re-run the cell below. This uses the same key, same rules, same tag — just for the listed events.


In [ ]:
# Manually list any failed events here
rerun_events = []  # e.g., ["cyclone_idai_2019"]
rerun_rules_name = "RULES1"  # or "RULES2", "RULES3"

if rerun_events:
    rerun_rules_text = dict(RULES_VARIANTS)[rerun_rules_name]
    df_rerun = df_sources[df_sources["event"].isin(rerun_events)].copy()
    display(df_rerun)

    rerun_tag = f"{TAG_BASE}-{rerun_rules_name}-filtered"
    with use_api_key_env(API_KEY_ENV_VAR):
        print(f">>> Rerunning {rerun_events} with {rerun_rules_name}")
        df_rerun_results = run_list_single(df_rerun, rerun_rules_text, MODEL, tag=rerun_tag)
    display(df_rerun_results)
else:
    print("No events to re-run (rerun_events is empty).")
